In [1]:
import httpx
import numpy as np

# import polars as pl
from collections import defaultdict
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt

VESPA_URL = "http://localhost:8080"
BATCH_SIZE = 250
UNKNOWN_CUISINE_VALUES = {"missing cuisine", "unknown", "n/a", "none", ""}

In [2]:
# whole_nlp_df = pl.read_parquet(
#     "../data/processed/lemmafied_df.parquet.gzip", use_pyarrow=True
# )
# whole_nlp_df.head(3)

In [3]:
# whole_nlp_df.columns

In [ ]:
def fetch_all_tagged_embeddings() -> dict[str, list]:
    """
    Query Vespa once per known cuisine value, each capped at 1000 results
    (Vespa's default max), avoiding deep pagination entirely.
    """
    # First, discover which cuisine values exist
    resp = httpx.post(
        f"{VESPA_URL}/search/",
        json={
            "yql": 'select cuisine_str from recipe where cuisine_str matches ".+" limit 0',
            "yql.grouping": True,
        },
        timeout=30.0,
    )

    # Simpler approach: get distinct cuisines via a grouping query
    resp = httpx.post(
        f"{VESPA_URL}/search/",
        json={
            "yql": (
                "select cuisine_str from recipe "
                'where cuisine_str matches ".+" '
                "| all(group(cuisine_str) each(output(count())))"
            )
        },
        timeout=30.0,
    )
    resp.raise_for_status()
    data = resp.json()

    cuisines = []
    try:
        groups = data["root"]["children"][0]["children"][0]["children"]
        for g in groups:
            cuisines.append(g["value"])
    except (KeyError, IndexError):
        print("Grouping query failed, falling back to hardcoded cuisine check")
        return {}

    print(f"Found {len(cuisines)} distinct cuisines: {cuisines}")

    cuisine_embeddings = defaultdict(list)

    for cuisine in cuisines:
        if cuisine in UNKNOWN_CUISINE_VALUES:
            continue

        resp = httpx.post(
            f"{VESPA_URL}/search/",
            json={
                "yql": (
                    f'select embedding from recipe where cuisine_str contains "{cuisine}" limit 500'
                )
            },
            timeout=30.0,
        )
        if resp.status_code != 200:
            print(f"Skipping '{cuisine}': {resp.text}")
            continue

        hits = resp.json()["root"].get("children", [])
        for hit in hits:
            embedding = hit.get("fields", {}).get("embedding", {}).get("values")
            if embedding:
                cuisine_embeddings[cuisine].append(embedding)

        print(f"  {cuisine}: {len(cuisine_embeddings[cuisine])} embeddings")

    return cuisine_embeddings

In [5]:
# cuisine_embeddings = fetch_all_tagged_embeddings()

In [6]:
def compute_centroids(cuisine_embeddings: dict[str, list]) -> tuple[list, np.ndarray]:
    """
    Average embeddings within each cuisine to get one centroid vector.
    Filters out cuisines with too few examples to be meaningful.
    """
    MIN_RECIPES_PER_CUISINE = 20

    labels = []
    centroids = []

    for cuisine, embeddings in sorted(cuisine_embeddings.items()):
        if len(embeddings) < MIN_RECIPES_PER_CUISINE:
            print(
                f"Skipping '{cuisine}' — only {len(embeddings)} recipes (min {MIN_RECIPES_PER_CUISINE})"
            )
            continue
        centroid = np.mean(np.array(embeddings), axis=0)
        labels.append(f"{cuisine} (n={len(embeddings)})")
        centroids.append(centroid)

    return labels, np.array(centroids)

In [7]:
def plot_dendrogram(
    labels: list, centroids: np.ndarray, output_path: str = "cuisine_dendrogram.png"
):
    """
    Hierarchically cluster the centroids and plot a dendrogram.
    Average linkage + cosine-like distance (via correlation) tends to
    work well for high-dimensional embedding centroids.
    """
    Z = linkage(centroids, method="average", metric="cosine")

    plt.figure(figsize=(12, max(8, len(labels) * 0.3)))
    dendrogram(
        Z,
        labels=labels,
        orientation="right",
        leaf_font_size=9,
    )
    plt.title("Cuisine similarity dendrogram (from recipe embeddings)")
    plt.xlabel("Cosine distance")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    print(f"\nSaved dendrogram to {output_path}")
    print(f"Clustered {len(labels)} cuisines")

In [8]:
print("Fetching embeddings grouped by cuisine from Vespa...")
cuisine_embeddings = fetch_all_tagged_embeddings()

print(f"\nFound {len(cuisine_embeddings)} distinct cuisine tags")
for cuisine, embeddings in sorted(cuisine_embeddings.items(), key=lambda x: -len(x[1])):
    print(f"  {cuisine}: {len(embeddings)} recipes")

labels, centroids = compute_centroids(cuisine_embeddings)

if len(labels) < 3:
    print("Not enough cuisines with sufficient data to cluster meaningfully.")
    return

plot_dendrogram(labels, centroids)

Fetching embeddings grouped by cuisine from Vespa...
Fetched 250 documents so far...
Fetched 500 documents so far...
Fetched 750 documents so far...
Fetched 1000 documents so far...
Fetched 1250 documents so far...
Vespa error: {"root":{"id":"toplevel","relevance":1.0,"fields":{"totalCount":0},"errors":[{"code":3,"summary":"Illegal query","message":"Offset of 1250 requested, configured limit: 1000. See https://docs.vespa.ai/en/reference/api/query.html#native-execution-parameters"}]}}


HTTPStatusError: Client error '400 Bad Request' for url 'http://localhost:8080/search/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400